In [ ]:
import random ##Python’un random (rastgele) veri üretme kütüphanesini içe aktarır.

 

categories = ["Elektronik", "Giyim", "Kitap", "Kozmetik", "EvYasam"] ##Ürün kategorilerini içeren bir liste oluşturur.

 

with open("big_orders.txt", "w", encoding="utf-8") as f:
    ##big_orders.txt adlı bir dosya açar.
    ##"w" → yazma modu (dosya yoksa oluşturur, varsa silip yeniden yazar).
    ##encoding="utf-8" → Türkçe karakterlerin düzgün yazılması için.
    ##with kullanımı dosya iş bitince otomatik kapanmasını sağlar.
    ##f → dosya nesnesi (yazma işlemlerini bununla yapacağız).

    for i in range(1000): ##0’dan 999’a kadar dönen bir döngü oluşturur.

        order_id = 1000 + i 

        customer = random.choice(["Ali", "Ayse", "Can", "Elif", "", "Mehmet"]) ##Rastgele bir müşteri adı seçer.Dikkat: "" (boş string) de var → bazı kayıtlar eksik müşteri adı içerecek.

        category = random.choice(categories + ["giyim", "ELEKTRONIK"]) ##Bu sayede veri tutarsız (dirty data) hale gelir.

        amount = random.choice([120, 450, 900, "", "abc", 1500, 2500]) 
##Sipariş tutarı rastgele seçilir.
##Ama burada bilinçli hatalar var:
##"" → boş değer
##"abc" → sayısal olmayan değer
##Yani veri temizleme (data cleaning) için test verisi.

        status = random.choice(["OK", "ERROR"]) ##Sipariş durumu belirlenir.

 

        row = f"{order_id},{customer},{category},{amount},{status}" ##Tüm verileri tek satırda birleştirir.

        f.write(row + "\n") ##Oluşturulan satırı dosyaya yazar.

 

print("1000 kayıtlık ham veri oluşturuldu.") 

1000 kayıtlık ham veri oluşturuldu.


In [ ]:
missing_customer = 0 ##müşteri adı boş olanları sayar

bad_amount = 0  ##hatalı tutarları sayar

bad_category = 0 ##geçersiz kategori sayısı

 

valid_categories = ["Elektronik", "Giyim", "Kitap", "Kozmetik", "EvYasam"] ##Bunun dışındakiler hatalı sayılacak

 

with open("big_orders.txt", "r", encoding="utf-8") as f: ##Dosyayı bu sefer "r" (read = okuma) modunda açıyoruz

    for line in f: ##Dosyayı satır satır gezer

        parts = line.strip().split(",") #strip() → satır başı/sonu boşluk ve \n temizler
        #split(",") → virgüle göre parçalar

#Liste içinden alanları alıyoruz
#index mantığı:
#0 → order_id
#1 → customer
#2 → category
#3 → amount
#4 → status

        customer = parts[1] 

        category = parts[2] 

        amount = parts[3] 

 

        if customer == "": 

            missing_customer += 1 #Müşteri adı boşsa sayacı artırır

 

        if amount == "" or not amount.isdigit():
             #amount == ""boş değer yakalar
            #not amount.isdigit()
            #sadece rakam mı diye kontrol eder"450" → True"abc" → False


            bad_amount += 1#Eğer yukarıdaki şartlardan biri doğruysa hatalı sayılır
           

 

        if category not in valid_categories: #Kategori listede yoksa hatalıdır

            bad_category += 1 #Hatalı kategori sayacı artar

 

print("Eksik müşteri:", missing_customer) 

print("Hatalı tutar:", bad_amount) 

print("Standart dışı kategori:", bad_category) 

Eksik müşteri: 163
Hatalı tutar: 277
Standart dışı kategori: 274


In [ ]:
clean_rows = [] #Temizlenmiş verileri tutmak için boş bir liste oluşturulur.

 

with open("big_orders.txt", "r", encoding="utf-8") as f: #Ham veriyi okuma modunda açıyoruz.

    for line in f: #Satırı temizleyip parçalarına ayırıyoruz

        parts = line.strip().split(",") 

 

        order_id = parts[0] #Sipariş ID aynen alınır değişiklik yok

        customer = parts[1] if parts[1] != "" else "Bilinmiyor" #Eğer müşteri boş değilse → olduğu gibi al.Eğer boşsa → "Bilinmiyor" yap

        category = parts[2].capitalize() #İlk harfi büyük yapar, geri kalanları küçük yapar

        amount = parts[3] 
#Tutar ve durum alınır
        status = parts[4] 

 

        if amount == "" or not amount.isdigit(): 
#Hatalı tutarlar düzeltilir
            amount = "0" 

 

        row = f"{order_id},{customer},{category},{amount},{status}" #Temizlenmiş veri tekrar birleştirilir

        clean_rows.append(row) #listeye ekleme.Her temiz satır listeye eklenir

 

with open("clean_big_orders.txt", "w", encoding="utf-8") as f: #Yeni bir dosya oluşturulur (temiz veri için)

    for row in clean_rows: #Listedeki tüm temiz veriler dosyaya yazılır

        f.write(row + "\n") 

 

print("Temiz veri hazır.") 

Temiz veri hazır.


In [ ]:
new_rows = [] #Yeni oluşturulacak (özellik eklenmiş) verileri tutan liste

 

with open("clean_big_orders.txt", "r", encoding="utf-8") as f: #Temizlenmiş veriyi okuyoruz

    for line in f: 

        parts = line.strip().split(",") 

 

        amount = int(parts[3]) #String → integer çeviriyoruz.Çünkü artık matematiksel karşılaştırma yapacağız

        status = parts[4] 

 
#segmentasyon (sınıflandırma)
#Siparişleri tutarına göre grupluyorsun:
        if amount >= 1500: 

            level = "Premium" 

        elif amount >= 500: 

            level = "Standart" 

        else: 

            level = "Basic" 

 

        is_error = 1 if status == "ERROR" else 0 #Binary (0-1) feature

 

        row = line.strip() + f",{level},{is_error}" #Mevcut satıra yeni sütunlar eklenir

        new_rows.append(row) #Yeni satır listeye eklenir

 

with open("featured_big_orders.txt", "w", encoding="utf-8") as f: 

    for row in new_rows: #Tüm veriler dosyaya yazılır

        f.write(row + "\n") 

 

print("Feature engineering tamamlandı.")

Feature engineering tamamlandı.


In [ ]:
import time #Kodun ne kadar sürede çalıştığını ölçmek için kullanılır

 

# Tüm dosyayı tek seferde okuma 

start = time.time() #Kodun başlangıç zamanını alır

 

total = 0 #Satır sayısını saymak için sayaç

with open("featured_big_orders.txt", "r", encoding="utf-8") as f: 

    for line in f: 

        total += 1 
#Dosyadaki tüm satırları tek tek okur
#Her satırda sayaç artırılır
#👉 Sonuç: toplam kayıt sayısı

 

end = time.time() #Kodun bitiş zamanı

print("Tek dosya okuma süresi:", round(end - start, 4), "sn") 
#end - start → geçen süre (saniye)
#round(..., 4) → 4 basamaklı yuvarlama

 

# Partition mantığı (kategori bazlı ayırma) 

start = time.time() #Tekrar süre ölçümü başlatılır

 

files = { 

    "Elektronik": 0,                  #dictionary (sözlük)Anahtar (key) → kategori.
                                      #Değer (value) → o kategoride kaç kayıt var
    "Giyim": 0, 

    "Kitap": 0, 

    "Kozmetik": 0, 

    "Evyasam": 0 

} 


 

with open("featured_big_orders.txt", "r", encoding="utf-8") as f: 

    for line in f: 

        parts = line.strip().split(",") 

        category = parts[2] #Satırdan kategori alınır

        if category in files: #Dictionary içinde kontrol

            files[category] += 1 #O kategoriye ait sayaç artar

 

end = time.time() #Süre ölçümü biter

print("Partition bazlı işlem süresi:", round(end - start, 4), "sn") 

print(files) 

Tek dosya okuma süresi: 0.0093 sn
Partition bazlı işlem süresi: 0.0023 sn
{'Elektronik': 282, 'Giyim': 287, 'Kitap': 151, 'Kozmetik': 147, 'Evyasam': 133}


In [ ]:
category_counts = {} #kategori bazlı sayım için boş dictionary

premium_count = 0 #Premium sipariş sayısı

error_count = 0 #toplam hata sayısı

 

with open("featured_big_orders.txt", "r", encoding="utf-8") as f: 

    for line in f: 

        parts = line.strip().split(",") 

 

        category = parts[2] #kategoriler

        level = parts[5] #Basic / Standart / Premium

        is_error = int(parts[6]) #0 veya 1 → string'i tekrar sayıya çeviriyoruz

 

        category_counts[category] = category_counts.get(category, 0) + 1 
#get() kullanımı.Bu satırın yaptığı şey:
#Eğer kategori dictionary’de varsa → değerini al
#Yoksa → 0 kabul et
#Sonra +1 ekle

 

        if level == "Premium": #Premium siparişleri sayar

            premium_count += 1 

 

        error_count += is_error #is_error zaten 0 veya 1 direkt topluyorsun
#👉 Yani:
#1 → hata,0 → hata değil
#✔️ Böylece ayrı if yazmana gerek yok

 

print("Kategori dağılımı:") 

for k, v in category_counts.items(): #.items()Dictionary’i key-value olarak döner

    print(k, "->", v) 

 

print("Premium sipariş sayısı:", premium_count) 

print("Toplam hata:", error_count) 

Kategori dağılımı:
Elektronik -> 282
Giyim -> 287
Kitap -> 151
Evyasam -> 133
Kozmetik -> 147
Premium sipariş sayısı: 287
Toplam hata: 509
